# * Diamond Data Source

In [45]:
import configparser
import datetime as dt
import pandas as pd
import numpy as np
import xlrd
import oracledb
import re

config = configparser.ConfigParser()
config.read('../../my_config.ini')
config.sections()

TDMDBPR_user = config['TDMDBPR']['username']
TDMDBPR_pwd = config['TDMDBPR']['password']
TDMDBPR_db = config['TDMDBPR']['db']
TDMDBPR_host = config['TDMDBPR']['host']
TDMDBPR_port = config['TDMDBPR']['port']

AKPIPRD_user = config['AKPIPRD']['username']
AKPIPRD_pwd = config['AKPIPRD']['password']
AKPIPRD_db = config['AKPIPRD']['db']
AKPIPRD_host = config['AKPIPRD']['host']
AKPIPRD_port = config['AKPIPRD']['port']

curr_dt = dt.datetime.now().date()
str_curr_dt = curr_dt.strftime('%Y%m%d')

In [46]:
''' Input parameter '''

op_dir = '../../data/Adhoc/output'
op_file = f'src_diamond_{str_curr_dt}'

print(f'\nParameter input...\n')
print(f'   -> op_dir: {op_dir}')
print(f'   -> op_file: {op_file}')


Parameter input...

   -> op_dir: ../../data/Adhoc/output
   -> op_file: src_diamond_20260318


## Import Transaction
-> STG_KPI_NEWCO_DIAMOND_ACTUAL_INC

In [47]:
''' Execute transaction '''


# Input parameter
# v_start_date = 20260101
print(f'\nParameter input...')
# print(f'   -> v_start_date: {v_start_date}')

curr_datetime = dt.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')
print(f'\nData as of {curr_datetime}')


# Connect : TDMDBPR
src_dsn = f'{TDMDBPR_user}/{TDMDBPR_pwd}@{TDMDBPR_host}:{TDMDBPR_port}/{TDMDBPR_db}'
src_conn = oracledb.connect(src_dsn)
src_cur = src_conn.cursor()

query = (f"""
    SELECT /*+PARALLEL(8)*/ 
        PAR_KEY, BASENAME, MAX(HD_LOADDATE) AS HD_LOADDATE
        , SUBSTR(TM_KEY_DAY,1,4) AS TM_KEY_YR
        , MIN(TM_KEY_DAY) AS MIN_DAY, MAX(TM_KEY_DAY) AS MAX_DAY
        , CASE 	WHEN REGEXP_LIKE(A.METRIC_NAME, 'Subs Share') THEN 'Subs Share' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'Gross Adds Share') THEN 'Gross Add Share' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'New Revenue|Existing Revenue|Other Revenue') THEN 'New & Existing & Other' 
                WHEN REGEXP_LIKE(LOWER(A.METRIC_NAME), 'usage sub|active sub|subbase') THEN 'Subs' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'Revenue') THEN 'Revenue' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'AP 1D') THEN 'AP 1D' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'Inflow') THEN 'Inflow' 
                WHEN REGEXP_LIKE(LOWER(A.METRIC_NAME), 'gross add|new sub|activation sub') THEN 'Gross Add' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'Churn') THEN 'Churn' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, '60DPD') THEN '60DPD' 
                WHEN REGEXP_LIKE(LOWER(A.METRIC_NAME), 'net add') THEN 'Net Add' 
                ELSE '' 
                END METRIC_GRP
        , A.METRIC_CD, A.METRIC_NAME
        , B.METRIC_NAME AS VIS_NAME
        , COMP_CD, VERSION
        , SUM(CASE WHEN AREA_TYPE = 'P' THEN METRIC_VALUE END) AS P_ACTUAL
        , SUM(CASE WHEN AREA_TYPE = 'G' THEN METRIC_VALUE END) AS G
        , SUM(CASE WHEN AREA_TYPE = 'H' THEN METRIC_VALUE END) AS H
        , SUM(CASE WHEN AREA_TYPE = 'HH' THEN METRIC_VALUE END) AS HH
        , SUM(CASE WHEN AREA_TYPE = 'CCAA' THEN METRIC_VALUE END) AS CCAA
        , SUM(CASE WHEN AREA_TYPE = 'CCAATT' THEN METRIC_VALUE END) AS CCAATT
        , MAX(LOAD_DATE) AS LOAD_DATE
    FROM GEOSPCAPPO.STG_KPI_NEWCO_DIAMOND_ACTUAL_INC A
    LEFT JOIN (
        SELECT DISTINCT METRIC_CD, METRIC_NAME, METRIC_GRP
        FROM GEOSPCAPPO.AGG_PERF_NEWCO
        WHERE AREA_CD = 'P'
        AND TM_KEY_DAY >= 20260101
    ) B
        ON B.METRIC_CD = A.METRIC_CD
    GROUP BY PAR_KEY, BASENAME, SUBSTR(TM_KEY_DAY,1,4), A.METRIC_CD, A.METRIC_NAME, B.METRIC_NAME, COMP_CD, VERSION
    --ORDER BY PAR_KEY, BASENAME, 3, A.METRIC_CD
""")


try:
    # Create Dataframe
    src_cur.execute(query)
    rows = src_cur.fetchall()
    src_df = pd.DataFrame.from_records(rows, columns=[x[0] for x in src_cur.description])
    print(f'\nDataFrame: {src_df.shape[0]} rows, {src_df.shape[1]} columns')

    # # Generate CSV file
    # src_df.to_csv(f'{op_dir}/{op_file}.csv', index=False, encoding='utf-8')
    # print(f'\n   -> Generate "{op_file}.csv" successfully')

    src_cur.close()


except oracledb.DatabaseError as e:
    print(f'\nError with Oracle : {e}')


finally:
    src_conn.close()


Parameter input...

Data as of 2026-03-18, 16:09:56

DataFrame: 159 rows, 19 columns


## Review

In [48]:
src_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   PAR_KEY      159 non-null    int64         
 1   BASENAME     157 non-null    object        
 2   HD_LOADDATE  159 non-null    datetime64[ns]
 3   TM_KEY_YR    159 non-null    object        
 4   MIN_DAY      159 non-null    int64         
 5   MAX_DAY      159 non-null    int64         
 6   METRIC_GRP   147 non-null    object        
 7   METRIC_CD    159 non-null    object        
 8   METRIC_NAME  159 non-null    object        
 9   VIS_NAME     85 non-null     object        
 10  COMP_CD      159 non-null    object        
 11  VERSION      159 non-null    object        
 12  P_ACTUAL     159 non-null    float64       
 13  G            156 non-null    float64       
 14  H            159 non-null    float64       
 15  HH           159 non-null    float64       
 16  CCAA    

In [49]:
missing_count = src_df.isnull().sum()
missing_count

PAR_KEY          0
BASENAME         2
HD_LOADDATE      0
TM_KEY_YR        0
MIN_DAY          0
MAX_DAY          0
METRIC_GRP      12
METRIC_CD        0
METRIC_NAME      0
VIS_NAME        74
COMP_CD          0
VERSION          0
P_ACTUAL         0
G                3
H                0
HH               0
CCAA           159
CCAATT         159
LOAD_DATE        0
dtype: int64

In [50]:
# src_df[['METRIC_GRP', 'METRIC_CD', 'METRIC_NAME']].drop_duplicates()

src_df['METRIC_GRP'].drop_duplicates()

0                   Gross Add
2                       Churn
3             Gross Add Share
5                  Subs Share
7                       AP 1D
8                      Inflow
17                      60DPD
19     New & Existing & Other
20                       None
39                    Revenue
110                   Net Add
113                      Subs
Name: METRIC_GRP, dtype: object

In [51]:
obsolete_df = src_df.query(" METRIC_GRP in ('Inflow','Gross Add','AP 1D','New & Existing & Other') ")
obsolete_df

,PAR_KEY,BASENAME,HD_LOADDATE,TM_KEY_YR,MIN_DAY,MAX_DAY,METRIC_GRP,METRIC_CD,METRIC_NAME,VIS_NAME,COMP_CD,VERSION,P_ACTUAL,G,H,HH,CCAA,CCAATT,LOAD_DATE
0,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Gross Add,DB1S000101,Prepay Activation sub,None,DTAC,A,2.090670e+05,2.090670e+05,2.090160e+05,2.090160e+05,None,None,2026-03-16
1,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Gross Add,DB1S000101AJ,Prepaid Gross Add : DTAC : Retail Sales,None,DTAC,A,1.084610e+05,1.167530e+05,1.084610e+05,1.084610e+05,None,None,2026-03-16
7,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,AP 1D,DB1S000109,Prepaid VAS Day1 / AP 1D : DTAC,None,DTAC,A,4.165556e+07,4.164893e+07,4.166135e+07,4.166135e+07,None,None,2026-03-16
8,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Inflow,DB2R000500AH,Postpaid Inflow M1 : DTAC : Others,None,DTAC,A,1.118044e+05,1.210094e+05,1.118044e+05,1.118044e+05,None,None,2026-03-16
9,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Inflow,DB1R000900AK,Prepaid Inflow M1 : DTAC : Wholesales,None,DTAC,A,2.707570e+05,2.703112e+05,2.703112e+05,2.703112e+05,None,None,2026-03-16
10,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Inflow,DB1R000900AE,Prepaid Inflow M1 : DTAC : Direct Sales,None,DTAC,A,1.054985e+06,1.054985e+06,1.054985e+06,1.054985e+06,None,None,2026-03-16
11,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Gross Add,DB2S000100AB,Postpaid Gross Add : DTAC : B2B,None,DTAC,A,1.958000e+03,2.036000e+03,3.372000e+03,3.372000e+03,None,None,2026-03-16
13,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Inflow,DB2R000500AI,Postpaid Inflow M1 : DTAC : Own Digital,None,DTAC,A,1.629341e+04,2.112785e+04,1.629341e+04,1.629341e+04,None,None,2026-03-16
14,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Gross Add,DB1S000101AG,Prepaid Gross Add : DTAC : Modern Trade,None,DTAC,A,2.635300e+04,2.812800e+04,2.676100e+04,2.676100e+04,None,None,2026-03-16
19,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,New & Existing & Other,DB1R000102,Prepay Existing Revenue,None,DTAC,A,5.159421e+08,5.159421e+08,5.156402e+08,5.156402e+08,None,None,2026-03-16


In [52]:
src_df

,PAR_KEY,BASENAME,HD_LOADDATE,TM_KEY_YR,MIN_DAY,MAX_DAY,METRIC_GRP,METRIC_CD,METRIC_NAME,VIS_NAME,COMP_CD,VERSION,P_ACTUAL,G,H,HH,CCAA,CCAATT,LOAD_DATE
0,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Gross Add,DB1S000101,Prepay Activation sub,None,DTAC,A,209067.0000,209067.0000,209016.0000,209016.0000,None,None,2026-03-16
1,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Gross Add,DB1S000101AJ,Prepaid Gross Add : DTAC : Retail Sales,None,DTAC,A,108461.0000,116753.0000,108461.0000,108461.0000,None,None,2026-03-16
2,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Churn,DB1S000300,Prepay Churn Rate,Prepaid Churn Rate : DTAC,DTAC,A,22.4900,149.2600,142713.0500,142713.0100,None,None,2026-03-16
3,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260101,20260315,Gross Add Share,VIN00105,Postpaid Gross Adds Share : AIS,Postpaid Gross Adds Share : AIS,ALL,A,2653.4377,21461.2327,168472.0649,257241.8443,None,None,2026-03-16
4,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260101,20260315,Gross Add Share,VIN00102,Mobile Gross Adds Share : DTAC,Mobile Gross Adds Share : DTAC,DTAC,A,1488.7333,11619.6062,92625.5249,132429.8705,None,None,2026-03-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Inflow,DB1R001000AH,Prepaid Inflow M2 : DTAC : Others,None,DTAC,A,22351.3946,22351.3946,22351.3946,22351.3946,None,None,2026-03-16
155,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Inflow,DB1R000900AH,Prepaid Inflow M1 : DTAC : Others,None,DTAC,A,101932.0621,101932.0621,101932.0621,101932.0621,None,None,2026-03-16
156,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,Gross Add,DB2S000100,Postpaid activation sub,None,DTAC,A,12243.0000,12243.0000,12141.0000,12141.0000,None,None,2026-03-16
157,20260316,D_CORPKPI_ACTUAL_01_20260316.txt.gpg,2026-03-17 23:14:12.505,2026,20260307,20260316,None,DB2S000104,MNP : DTAC,None,DTAC,A,289.0000,311.0000,289.0000,289.0000,None,None,2026-03-16


## Output

In [53]:
''' Generate CSV file '''

src_df.to_csv(f'{op_dir}/{op_file}.csv', index=False, encoding='utf-8')
print(f'\n   -> Generate "{op_file}.csv" successfully')


   -> Generate "src_diamond_20260318.csv" successfully
